# `dbspend360_pool_spends_app`

Per-pool spend rollup for **Databricks instance pools**. Sibling pipeline
to `dbspend360_cluster_spends_app`, but keyed on `instance_pool_id`.

Headline metric: **idle VM cost** = pool VMs that incur cloud spend with
no cluster attached (computed by subtraction, locked in by §4.5 #1).

Sources:
- `dbspend360_cloud_cost_explorer` — for `pool_total_cost` (rows where
  `instance_pool_id IS NOT NULL`) and the per-cluster cloud cost used
  to back out `active_cloud_cost`.
- `system.compute.clusters` — `worker_instance_pool_id` /
  `driver_instance_pool_id` join (deduped to one row per cluster) to
  attach clusters to pools.
- `system.billing.usage` × `system.billing.list_prices` — DBU cost
  attributed by `usage_metadata.instance_pool_id` (typically near
  zero on non-premium editions).

Merge key: `(instance_pool_id, workspace_id, usage_date)`.

Widgets mirror the cluster spends notebook: `catalog`, `schema`,
`overlap_days`, `workspace_ids`.

In [ ]:
%run ./utils_common

In [ ]:
logger = setup_logger('PoolSpendsReporter')

In [ ]:
dbutils.widgets.text('catalog', '', 'CATALOG')
dbutils.widgets.text('schema', '', 'SCHEMA')
dbutils.widgets.text('overlap_days', '3', 'Overlap days (min 2)')
dbutils.widgets.text('workspace_ids', '', 'Workspace IDs (comma-separated, blank=all)')

In [ ]:
# =======================================================
# Pool Spends Client
# =======================================================
class PoolSpendsClient:
  TABLE_NAME = 'dbspend360_pool_spends'

  # Ceiling on the negative-residual rate. Tag-collection lag in CE/CM
  # makes a small floor-fire rate unavoidable; if it crosses this
  # threshold the audit log surfaces a quality warning so an operator
  # investigates a persistent skew.
  IDLE_FLOOR_FIRE_WARN_PCT = 1.0

  def __init__(
    self,
    audit_table: str,
    cloud_cost_table: str,
    target_table: str,
    error_log_table: str,
    overlap_days: int,
    logger=None,
  ):
    self.audit_table = audit_table
    self.cloud_cost_table = cloud_cost_table
    self.target_table = target_table
    self.error_log_table = error_log_table
    self.overlap_days = overlap_days
    self.logger = logger or logging.getLogger('PoolSpendsClient')

    raw_ws = dbutils.widgets.get('workspace_ids')
    if raw_ws.strip() == '':
      self.workspace_ids = None
    else:
      self.workspace_ids = [w.strip() for w in raw_ws.split(',') if w.strip()]

  def _cluster_to_pool_df(self):
    """One row per (cluster_id, instance_pool_id) attachment.

    `system.compute.clusters` is a slowly-changing snapshot, so we
    dedup by (cluster_id, pool) before the join. The cluster's pool
    is read from `worker_instance_pool_id` first, falling back to
    `driver_instance_pool_id` so single-driver pools still attach.
    """
    df = spark.table('system.compute.clusters').filter(
      F.col('worker_instance_pool_id').isNotNull() | F.col('driver_instance_pool_id').isNotNull()
    )
    if self.workspace_ids is not None:
      df = df.filter(F.col('workspace_id').isin(self.workspace_ids))
    return (
      df.select(
        'cluster_id',
        F.coalesce(
          F.col('worker_instance_pool_id'),
          F.col('driver_instance_pool_id'),
        ).alias('instance_pool_id'),
      )
      .filter(F.col('instance_pool_id').isNotNull())
      .dropDuplicates(['cluster_id', 'instance_pool_id'])
    )

  def _cloud_cost_pool_total_df(self, start_dt, end_dt):
    """Per-(instance_pool_id, usage_date) raw pool-tag total.

    Reads pool-tagged rows (cluster_id IS NULL, instance_pool_id IS NOT NULL)
    landed by the parallel CE/CM tag query path added in slice 3.
    """
    return (
      spark.table(self.cloud_cost_table)
      .filter(F.col('instance_pool_id').isNotNull())
      .filter(
        (F.col('cost_incurred_date') >= F.lit(start_dt))
        & (F.col('cost_incurred_date') <= F.lit(end_dt))
      )
      .groupBy('instance_pool_id', 'cost_incurred_date')
      .agg(
        F.sum('cloud_cost').alias('pool_total_cost'),
        F.first('currency', ignorenulls=True).alias('currency'),
      )
    )

  def _cloud_cost_attached_df(self, start_dt, end_dt, cluster_to_pool_df):
    """Per-(instance_pool_id, usage_date) sum of attached-cluster cloud cost.

    Joins cluster-tagged rows in `dbspend360_cloud_cost_explorer` to the
    cluster→pool attachment table; the SUM is what gets subtracted from
    `pool_total_cost` to derive `idle_cloud_cost`.
    """
    cluster_costs = (
      spark.table(self.cloud_cost_table)
      .filter(F.col('cluster_id').isNotNull())
      .filter(
        (F.col('cost_incurred_date') >= F.lit(start_dt))
        & (F.col('cost_incurred_date') <= F.lit(end_dt))
      )
      .select('cluster_id', 'cost_incurred_date', 'cloud_cost')
    )
    return (
      cluster_costs.alias('cc')
      .join(cluster_to_pool_df.alias('m'), on='cluster_id', how='inner')
      .groupBy('instance_pool_id', 'cost_incurred_date')
      .agg(F.sum('cloud_cost').alias('active_cloud_cost'))
    )

  def _pool_dbu_costs_df(self, start_dt, end_dt):
    """Per-(instance_pool_id, workspace_id, usage_date) DBU $.

    Filters `system.billing.usage` to rows where
    `usage_metadata.instance_pool_id IS NOT NULL`. No
    `cluster_source = 'JOB'` join filter — pool DBU is owned by the
    pool, not by a cluster.
    """
    usage_df = (
      spark.table('system.billing.usage')
      .alias('usage')
      .filter(
        (F.col('usage.usage_date') >= F.lit(start_dt))
        & (F.col('usage.usage_date') <= F.lit(end_dt))
      )
      .filter(F.col('usage.usage_metadata')['instance_pool_id'].isNotNull())
    )
    if self.workspace_ids is not None:
      usage_df = usage_df.filter(F.col('usage.workspace_id').isin(self.workspace_ids))

    list_prices_df = spark.table('system.billing.list_prices').alias('list_prices')

    priced = usage_df.join(
      list_prices_df,
      on=(
        (F.col('usage.sku_name') == F.col('list_prices.sku_name'))
        & (F.col('usage.usage_start_time') >= F.col('list_prices.price_start_time'))
        & (
          (F.col('usage.usage_start_time') < F.col('list_prices.price_end_time'))
          | F.col('list_prices.price_end_time').isNull()
        )
      ),
      how='left',
    )

    return priced.groupBy(
      F.col('usage.usage_metadata')['instance_pool_id'].alias('instance_pool_id'),
      F.col('usage.workspace_id').alias('workspace_id'),
      F.col('usage.usage_date').alias('usage_date'),
    ).agg(
      F.sum(
        F.col('usage.usage_quantity') * F.col('list_prices.pricing')['default'].cast('double')
      ).alias('databricks_cost'),
      F.lit('USD').alias('currency'),
    )

  def _pool_workspace_df(self):
    """Best-effort lookup from `instance_pool_id` to `workspace_id`.

    Cloud cost rows do not carry `workspace_id` (the cost-explorer table
    is keyed on the cloud-tag value alone). We borrow it from
    `system.compute.clusters`, where each pool that is attached to at
    least one cluster has a known workspace. Pools never attached to
    any cluster (orphans) get NULL workspace_id and the read path
    renders 'pool name unknown'.
    """
    df = spark.table('system.compute.clusters').filter(
      F.col('worker_instance_pool_id').isNotNull() | F.col('driver_instance_pool_id').isNotNull()
    )
    if self.workspace_ids is not None:
      df = df.filter(F.col('workspace_id').isin(self.workspace_ids))
    return (
      df.select(
        F.coalesce(
          F.col('worker_instance_pool_id'),
          F.col('driver_instance_pool_id'),
        ).alias('instance_pool_id'),
        F.col('workspace_id'),
      )
      .filter(F.col('instance_pool_id').isNotNull())
      .dropDuplicates(['instance_pool_id', 'workspace_id'])
    )

  def _log_floor_fire_rate(self, joined_df):
    """Audit log when the `greatest(..., 0)` floor fires too often.

    Negative residuals are expected occasionally because CE/CM may
    surface a `ClusterId` row before the corresponding
    `InstancePoolId` row catches up. A persistent skew (> 1% of rows)
    suggests a tag-collection problem and should surface in the audit
    log so an operator notices.
    """
    metrics = joined_df.agg(
      F.count('*').alias('total_rows'),
      F.sum(
        F.when(F.col('pool_total_cost') - F.col('active_cloud_cost') < 0, 1).otherwise(0)
      ).alias('negative_rows'),
    ).collect()[0]
    total = int(metrics.total_rows or 0)
    negative = int(metrics.negative_rows or 0)
    if total == 0:
      return 'idle_floor_fire_rate=0/0'
    pct = (negative / total) * 100.0
    msg = f'idle_floor_fire_rate={negative}/{total} ({pct:.2f}%)'
    if pct > self.IDLE_FLOOR_FIRE_WARN_PCT:
      self.logger.warning(
        f'Idle-cost negative residual rate {pct:.2f}% exceeds '
        f'{self.IDLE_FLOOR_FIRE_WARN_PCT}% — investigate CE/CM tag-collection lag.'
      )
      try:
        write_error_log_entries(
          [msg],
          'POOL_SPENDS',
          'IDLE_FLOOR_FIRE',
          self.error_log_table,
        )
      except Exception as e:
        self.logger.warning(f'Failed to write floor-fire audit entry: {e}')
    return msg

  def compute_and_merge_pool_spends(self):
    start_dt = end_dt = datetime.now(timezone.utc).date()
    try:
      start_dt, end_dt = get_date_window(self.audit_table, self.TABLE_NAME, self.overlap_days)

      valid, msg = validate_date_window(start_dt, end_dt)
      if not valid:
        raise DataQualityError(msg)

      self.logger.info(f'Building dbspend360_pool_spends for {start_dt} \u2192 {end_dt}')

      pool_total_df = self._cloud_cost_pool_total_df(start_dt, end_dt)
      cluster_to_pool_df = self._cluster_to_pool_df()
      attached_df = self._cloud_cost_attached_df(start_dt, end_dt, cluster_to_pool_df)
      dbu_df = self._pool_dbu_costs_df(start_dt, end_dt)
      workspace_lookup = self._pool_workspace_df()

      pool_total_present = pool_total_df.limit(1).count() > 0
      dbu_present = dbu_df.limit(1).count() > 0

      if not pool_total_present and not dbu_present:
        self.logger.info('No pool spend signal in this date window; nothing to merge.')
        log_audit_run(
          self.audit_table,
          self.TABLE_NAME,
          start_dt,
          end_dt,
          'SUCCESS',
          0,
          'No pool data in window',
        )
        return

      # Cloud-cost subtraction: pool_total - sum(attached cluster costs).
      # `greatest(..., 0)` guards against tag-collection lag (see plan).
      cloud_joined = (
        pool_total_df.alias('p')
        .join(
          attached_df.alias('a'),
          on=['instance_pool_id', 'cost_incurred_date'],
          how='left',
        )
        .withColumn(
          'active_cloud_cost',
          F.coalesce(F.col('active_cloud_cost'), F.lit(0.0)),
        )
      )
      cloud_joined = safe_cache(cloud_joined)
      floor_msg = self._log_floor_fire_rate(cloud_joined)

      cloud_signal = (
        cloud_joined.withColumn(
          'idle_cloud_cost',
          F.greatest(
            F.col('pool_total_cost') - F.col('active_cloud_cost'),
            F.lit(0.0),
          ),
        )
        .withColumnRenamed('cost_incurred_date', 'usage_date')
        .select(
          'instance_pool_id',
          'usage_date',
          'pool_total_cost',
          'active_cloud_cost',
          'idle_cloud_cost',
          'currency',
        )
      )

      # Outer-join cloud signal with DBU signal so pools that have one
      # but not the other still produce a row. workspace_id may be NULL
      # for cloud-only rows when the pool isn't currently attached to
      # any cluster — the read path handles that gracefully.
      cloud_keyed = (
        cloud_signal.alias('c')
        .join(
          workspace_lookup.alias('w'),
          on='instance_pool_id',
          how='left',
        )
        .select(
          F.col('c.instance_pool_id').alias('instance_pool_id'),
          F.col('w.workspace_id').alias('workspace_id'),
          F.col('c.usage_date').alias('usage_date'),
          F.col('c.pool_total_cost'),
          F.col('c.active_cloud_cost'),
          F.col('c.idle_cloud_cost'),
          F.col('c.currency'),
        )
      )

      joined = cloud_keyed.alias('c').join(
        dbu_df.alias('d'),
        on=['instance_pool_id', 'workspace_id', 'usage_date'],
        how='outer',
      )

      final_df = (
        joined.select(
          F.col('instance_pool_id'),
          F.col('workspace_id'),
          F.col('usage_date'),
          F.coalesce(F.col('pool_total_cost'), F.lit(0.0)).alias('pool_total_cost'),
          F.coalesce(F.col('active_cloud_cost'), F.lit(0.0)).alias('active_cloud_cost'),
          F.coalesce(F.col('idle_cloud_cost'), F.lit(0.0)).alias('idle_cloud_cost'),
          F.coalesce(F.col('databricks_cost'), F.lit(0.0)).alias('databricks_cost'),
          F.coalesce(F.col('c.currency'), F.col('d.currency'), F.lit('USD')).alias('currency'),
        )
        .withColumn(
          'total_cost',
          F.col('pool_total_cost') + F.col('databricks_cost'),
        )
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
      )
      final_df = safe_cache(final_df)
      safe_unpersist(cloud_joined)

      row_count = final_df.count()

      validate_source_schema(
        final_df,
        {
          'instance_pool_id': 'string',
          'usage_date': 'date',
          'pool_total_cost': 'double',
          'active_cloud_cost': 'double',
          'idle_cloud_cost': 'double',
          'databricks_cost': 'double',
        },
        self.target_table,
        self.logger,
      )
      validate_no_negative_costs(
        final_df,
        [
          'pool_total_cost',
          'active_cloud_cost',
          'idle_cloud_cost',
          'databricks_cost',
          'total_cost',
        ],
        self.target_table,
        self.logger,
      )
      validate_currency_consistency(final_df, 'currency', self.target_table, self.logger)

      target = DeltaTable.forName(spark, self.target_table)
      (
        target.alias('t')
        .merge(
          final_df.alias('s'),
          't.instance_pool_id = s.instance_pool_id '
          'AND t.workspace_id <=> s.workspace_id '
          'AND t.usage_date = s.usage_date',
        )
        .whenMatchedUpdate(
          set={
            'pool_total_cost': 's.pool_total_cost',
            'active_cloud_cost': 's.active_cloud_cost',
            'idle_cloud_cost': 's.idle_cloud_cost',
            'databricks_cost': 's.databricks_cost',
            'currency': 's.currency',
            'total_cost': 's.total_cost',
            'updated_at': 'current_timestamp()',
          }
        )
        .whenNotMatchedInsert(
          values={
            'instance_pool_id': 's.instance_pool_id',
            'workspace_id': 's.workspace_id',
            'usage_date': 's.usage_date',
            'pool_total_cost': 's.pool_total_cost',
            'active_cloud_cost': 's.active_cloud_cost',
            'idle_cloud_cost': 's.idle_cloud_cost',
            'databricks_cost': 's.databricks_cost',
            'currency': 's.currency',
            'total_cost': 's.total_cost',
            'created_at': 'current_timestamp()',
            'updated_at': 'current_timestamp()',
          }
        )
        .execute()
      )

      safe_unpersist(final_df)
      get_merge_metrics(self.target_table, self.logger)

      validate_post_merge(
        self.target_table,
        'usage_date',
        start_dt,
        end_dt,
        row_count,
        self.logger,
      )

      log_audit_run(
        self.audit_table,
        self.TABLE_NAME,
        start_dt,
        end_dt,
        'SUCCESS',
        row_count,
        floor_msg,
      )
      self.logger.info(
        f'Merged {row_count} rows into {self.target_table} for {start_dt} \u2192 {end_dt}.'
      )

    except Exception as e:
      msg = str(e)[:1000]
      self.logger.error(f'Run failed: {msg}')
      try:
        log_audit_run(
          self.audit_table,
          self.TABLE_NAME,
          start_dt,
          end_dt,
          'FAILED',
          0,
          msg,
        )
      except Exception:
        self.logger.error('Failed to write FAILED audit entry')
      raise

In [ ]:
# =======================================================
# APP
# =======================================================
class PoolSpendsReporterApp:
  def __init__(self):
    catalog = dbutils.widgets.get('catalog')
    schema = dbutils.widgets.get('schema')
    ov_days = get_overlap_days(dbutils.widgets.get('overlap_days'), logger=logger)

    self.client = PoolSpendsClient(
      audit_table=build_table_fqn(catalog, schema, 'dbspend360_audit_log'),
      cloud_cost_table=build_table_fqn(catalog, schema, 'dbspend360_cloud_cost_explorer'),
      target_table=build_table_fqn(catalog, schema, 'dbspend360_pool_spends'),
      error_log_table=build_table_fqn(catalog, schema, 'dbspend360_error_log'),
      overlap_days=ov_days,
      logger=logger,
    )

  def run(self):
    self.client.compute_and_merge_pool_spends()

In [ ]:
# =======================================================
# Execute
# =======================================================
app = PoolSpendsReporterApp()
app.run()